<p> <center> <a href="../start_here_ja.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="03_low_level_mcp_ja.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="01_inference_endpoint_ja.ipynb">1</a>
        <a href="02_introduction_mcp_ja.ipynb">2</a>
        <a href="03_low_level_mcp_ja.ipynb">3</a>
        <a >4</a>
        <a href="05_nemo_agent_toolkit_ja.ipynb">5</a>
        <a href="06_challenge_ja.ipynb">6</a>
        <a href="bonus_challenge/07_bonus_challenge_ja.ipynb">7</a>
    </span>
    <span style="float: left; width: 33%; text-align: right;"><a href="05_nemo_agent_toolkit_ja.ipynb">Next Notebook</a></span>
</div>

## 学習目標

このノートブックを終えると、以下ができるようになります：
- LangGraph の State schema を定義し、StateGraph・node・edge を使ったチャットボットワークフローを構築する
- `nvidia` model provider を使って NVIDIA NIM endpoint を LLM バックエンドとして接続する
- `graph.stream()` を使ってリアルタイム出力のためにグラフレスポンスをストリーミングする
- Pydantic モデルを使ったパース可能な LLM レスポンスのために structured output を実装する

## 環境のセットアップ

最初のノートブックで、生成した `NVIDIA API KEY` の設定方法を学びました。このノートブックの前提として、任意の NIMs docker image を pull するために `NVIDIA_API_KEY` という環境変数にキーを設定する必要があります。キーをまだ取得していない場合は、NVIDIA NIMs API の[ホームページ](https://build.nvidia.com/explore/discover)にアクセスして API Key を生成してください。下のセルを実行し、表示されるテキストボックスに `NVIDIA API KEY` を入力してからキーボードの Enter キーを押してください。

In [ ]:
import os
import getpass

if not os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    nvapi_key = getpass.getpass("Enter your NVIDIA API key: ")
    assert nvapi_key.startswith("nvapi-"), f"{nvapi_key[:5]}... is not a valid key"
    os.environ["NVIDIA_API_KEY"] = nvapi_key
    os.environ["NGC_API_KEY"] = nvapi_key

## LangGraph の概要

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages

In [ ]:
from langchain.chat_models import init_chat_model

これにより、NVIDIA NIM に接続した LangChain chat model が作成されます。`init_chat_model()` 関数がすべての設定を自動的に処理します—モデル ID と provider を指定するだけで、すぐにレスポンスを生成できます。

In [ ]:
MODEL_ID = 'meta/llama-3.2-3b-instruct'
llm = init_chat_model(model=MODEL_ID, model_provider="nvidia",max_tokens=32)

In [ ]:
# Comment this out if you are using the local endpoint with right local port
# LOCAL_CONTAINER_PORT = 11579
# llm = ChatNVIDIA(base_url="http://0.0.0.0:{}/v1".format(CONTAINER_PORT), model="meta/llama-3.2-3b-instruct")

In [ ]:
# llm.get_available_models()

このセクションでは、LangGraph の StateGraph を使ってシンプルな agentic workflow を構築します。手順は以下のとおりです：

1. state schema を持つ `StateGraph` を作成する
2. `add_node(name, function)` を使って node を追加する
3. `add_edge(source, target)` を使って edge を追加する
4. 実行前に graph を compile する

In [ ]:
class State(TypedDict):
    """
    Graph state schema.
    - messages: List of conversation messages with automatic append behavior
    """
    messages: Annotated[list, add_messages]

State は `add_messages` reducer を使って会話履歴を保持し、新しいメッセージをリストに自動的に追加します。

Node は state を受け取り、処理を実行し、更新された state を返す Python 関数です。

In [ ]:
def chatbot(state: State):
    """
    Chatbot node that invokes the LLM with conversation history.
    Returns updated state with the assistant's response.
    """
    return {"messages": [llm.invoke(state["messages"])]}

In [ ]:
graph_builder = StateGraph(State)

# Add the chatbot node
graph_builder.add_node("chatbot", chatbot)

# Connect START -> chatbot (entry point)
graph_builder.add_edge(START, "chatbot")

# Compile the graph
graph = graph_builder.compile()

### グラフの可視化

グラフを実行する前に、compile した LangGraph workflow を Mermaid diagram として描画します。これにより `START` node から `chatbot` node への接続が想定通りかをひと目で確認できます。

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

`graph.invoke()` を使って同期的に完全なレスポンスを取得する

In [ ]:
graph.invoke({"messages": [{"role": "user", "content": "What is chicago known for?"}]})

`graph.stream()` を使ってリアルタイムのトークンごとのレスポンスを取得し、ユーザー体験を向上させます。

In [ ]:
def stream_graph_updates(user_input: str):
    """Stream responses from the graph for real-time output."""
    for event in graph.stream({"messages": [{"role": "user", "content": user_input}]}):
        for value in event.values():
            print("Assistant:", value["messages"][-1].content)

In [ ]:
stream_graph_updates("what is portland known for?")

## Pydantic を使った Structured Output

アプリケーションでは、後続の処理のために LLM のレスポンスをパース可能なフォーマット（例：JSON）で受け取る必要があることがよくあります。NVIDIA NIM は guided JSON schema を使った structured generation をサポートしています。Pydantic の `BaseModel` を使って期待する出力構造を定義し、`Literal` 型で出力を特定の値に制限します。

In [ ]:
from pydantic import BaseModel
from typing import Literal

class UserIntent(BaseModel):
    """The user's current intent in the conversation"""
    intent: Literal["naruto", "bleach"]

参考: [NIM Structured Generation Docs](https://docs.nvidia.com/nim/large-language-models/latest/structured-generation.html)

In [ ]:
llm_structured = init_chat_model(model=MODEL_ID, model_provider="nvidia").with_structured_output(UserIntent, strict=True)

In [ ]:
# llm = ChatNVIDIA(base_url="http://0.0.0.0:{}/v1".format(CONTAINER_PORT), model=MODEL_ID).with_structured_output(UserIntent, strict=True)

`.with_structured_output()` を使って Pydantic schema を LLM レスポンスに適用します。

In [ ]:
# Test: Classify user intent based on anime question
res = llm_structured.invoke([
    {'role':'system','content':'You are an anime encyclopedia. Classify if the user is asking a question on naruto or bleach.'},
    {'role':'user','content':'who is sasuke?'}
])

In [ ]:
print(f'intent: {res}')

## Memory（メモリ）

AI アプリケーションでは、複数のインタラクションにまたがってコンテキストを共有するためにメモリが必要です。

LangGraph では 2 種類のメモリを追加できます。
1) 短期メモリ（スレッドレベルの永続化）— agent がマルチターン会話を追跡できるようにします。
2) 長期メモリ — ユーザー固有またはアプリケーション固有のデータを会話をまたいで保存します。

このチュートリアルとチャレンジでは短期メモリのみを使用します。

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver  
from langgraph.graph import StateGraph
import json
from langchain_core.messages import convert_to_openai_messages

checkpointer = InMemorySaver()  

graph = graph_builder.compile(checkpointer=checkpointer)  

res = graph.invoke(
    {"messages": [{"role": "user", "content": "what is cuda?"}]},
    {"configurable": {"thread_id": "1"}},
    
)

In [ ]:
json.dumps(convert_to_openai_messages(res['messages']))

Expected output:

```json
[{"role": "user", "content": "what is cuda?"}, {"role": "assistant", "content": "CUDA (Parallel Computation Engine) is a parallel computing platform and programming model developed by NVIDIA. It allows developers to harness the power of multiple graphics processing units ("}]
```

In [ ]:
res = graph.invoke(
    {"messages": [{"role": "user", "content": "what was my previous question?"}]},
    {"configurable": {"thread_id": "1"}},  
)

In [ ]:
json.dumps(convert_to_openai_messages(res['messages']))

Expected output:

```json
[{"role": "user", "content": "what is cuda?"}, {"role": "assistant", "content": "CUDA (Parallel Computation Engine) is a parallel computing platform and programming model developed by NVIDIA. It allows developers to harness the power of multiple graphics processing units ("}, {"role": "user", "content": "what was my previous question?"}, {"role": "assistant", "content": "Your previous question was: \\"what is cuda?\\""}]
```

同じスレッド ID（`{"configurable": {"thread_id": "1"}}`）で `graph.invoke` を実行することで、グラフは過去の会話を記録し、その履歴を利用して会話を継続できます。

## Interrupts（割り込み）

Interrupt により、グラフの実行を特定のポイントで一時停止し、外部入力を待ってから処理を再開できます。  
これにより、処理を進めるために外部入力が必要な human-in-the-loop パターンが実現します。  
Interrupt がトリガーされると、LangGraph は永続化レイヤーを使ってグラフの状態を保存し、再開されるまで無期限に待機します。

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt

class FormState(TypedDict):
    age: int | None

def dummy_start_node(state: FormState):
    print('start node')

def get_age_node(state: FormState):
    prompt = "What is your age?"

    while True:
        answer = interrupt(prompt)  # payload surfaces in result["__interrupt__"]

        if isinstance(answer, int) and answer > 0:
            return {"age": answer}

        prompt = f"'{answer}' is not a valid age. Please enter a positive number."

memory = InMemorySaver()

builder = StateGraph(FormState)
builder.add_node("dummy_start_node",dummy_start_node)
builder.add_node("collect_age", get_age_node)
builder.add_edge(START,"dummy_start_node")
builder.add_edge("dummy_start_node","collect_age")
builder.add_edge("collect_age", END)

graph = builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "form-1"}}
first = graph.invoke({"age": None}, config=config)
print(first["__interrupt__"])  # -> [Interrupt(value='What is your age?', ...)]

# Provide invalid data; the node re-prompts
retry = graph.invoke(Command(resume="thirty"), config=config)
print(retry["__interrupt__"])  # -> [Interrupt(value="'thirty' is not a valid age...", ...)]

# Provide valid data; loop exits and state updates
final = graph.invoke(Command(resume=30), config=config)
print(final["age"])  # -> 30

Expected output:

```
start node.  
[Interrupt(value='What is your age?', id='93589a0b323f03eeaa19f89000f5216c')].  
[Interrupt(value="'thirty' is not a valid age. Please enter a positive number.", id='93589a0b323f03eeaa19f89000f5216c')]. 
30
```

グラフは `dummy_start_node` から始まり、'start node' を出力します。

`graph.invoke({"age": None}, config=config)` では、`{"age": None}` がグラフの状態として渡されます。これにより 'What is your age?' という Interrupt が返されます。

`graph.invoke(Command(resume="thirty"), config=config)` では、最初の呼び出しの状態（`{"age": None}`）がそのまま保持されます。`Command(resume="thirty")` の値 "thirty" は `get_age_node` 内の変数 'answer' に返されます。

`graph.invoke(Command(resume=30), config=config)` では、上記と同様に状態は変化しません。`Command(resume=30)` の値 '30' は `get_age_node` 内の変数 'answer' に返され、Interrupt なしで最終値として返されます。

<b>重要：`graph.invoke(Command(resume=30), config=config)` を実行すると、Interrupt を発生させたノードが最初から完全に再実行されます。つまり、`get_age_node` 関数内のすべての処理（`prompt = "What is your age?"` から）が毎回再実行されます。</b>

## Agent Skills

Agent skill とは、agent が特定のタスクをより適切に実行するために動的に発見・読み込みできる命令、スクリプト、およびリソースのセットです。  
その核心として、skill は SKILL.md ファイルを含むフォルダーです。最低限、skill.md ファイルには 'name' と 'description' のメタデータフィールドが必要です。また、skill の機能を説明する命令も含めることができます。

Skill は progressive disclosure を使ってコンテキストを効率的に管理します。
* 発見（Discovery）：起動時に、agent は利用可能な各 skill の name と description のみを読み込みます。それだけで、いつ使えるかを把握するのに十分です。
* 有効化（Activation）：タスクが skill の description と一致すると、agent はその skill の命令を読み込みます。
* 実行（Execution）：agent は skill に記載された命令に従います。

MCP プロトコルでは入力スキーマ全体を最初からエージェントのコンテキストに含める必要があるのに対し、skill の場合は最初に name と description のみがコンテキストに渡され、命令は必要に応じてのみ読み込まれます。

Skill にはオプションでスクリプト、参照、アセットを含めることもできます。このチュートリアルとチャレンジでは省略します。

### Skills Implementation（スキル実装）

[Skill を agent に統合する方法は主に 2 つ](https://agentskills.io/integrate-skills)あります。

1. filesystem ベースの agent

コンピューター環境（bash/unix）内で動作し、最も高機能な選択肢です。`cat /path/to/my-skill/SKILL.md` のようなシェルコマンドで skill が有効化されます。バンドルされたリソースもシェルコマンドでアクセスします。

2. tool ベースの agent

専用のコンピューター環境なしに機能します。代わりに、モデルが skill をトリガーしたりバンドルされたアセットにアクセスしたりするための tool を実装します。具体的な tool の実装は開発者に委ねられます。

このチュートリアルとチャレンジでは filesystem ベースの agent に焦点を当てます。tool ベースの agent の実装については LangChain のドキュメントを参照してください。

`skills` フォルダーを作成します。これがすべての skill のディレクトリになります。このチュートリアルでは `sales-analytics` という 1 つの skill のみを使用します。  
これは [skills sql assistant](https://github.com/langchain-ai/docs/blob/77684f277bb731e35a58dff38f1295a69fb389ad/src/oss/langchain/multi-agent/skills-sql-assistant.mdx) を参考にしています。

In [ ]:
!mkdir -p skills/sales-analytics

[skills の標準仕様](https://agentskills.io/specification)に従って、

1. frontmatter に name と description を記入します。
2. frontmatter の後に instructions を記入します。[任意]

[ガイダンス](https://agentskills.io/specification#progressive-disclosure)によると、name と description フィールドはおよそ 100 トークンで、instructions は 5000 トークン未満が推奨されています。

標準 SKILL.md テンプレート
```
---
name: skill-name
description: A description of what this skill does and when to use it.
---
<instructions in markdown here>
```

In [ ]:
%%writefile skills/sales-analytics/SKILL.md
---
name: sales-analytics
description: Database schema and business logic for sales data analysis including customers, orders, and revenue.
---
# Sales Analytics Schema

## Tables

### customers
- customer_id (PRIMARY KEY)
- name
- email
- signup_date
- status (active/inactive)
- customer_tier (bronze/silver/gold/platinum)

### orders
- order_id (PRIMARY KEY)
- customer_id (FOREIGN KEY -> customers)
- order_date
- status (pending/completed/cancelled/refunded)
- total_amount
- sales_region (north/south/east/west)

### order_items
- item_id (PRIMARY KEY)
- order_id (FOREIGN KEY -> orders)
- product_id
- quantity
- unit_price
- discount_percent

## Business Logic

**Active customers**: status = 'active' AND signup_date <= CURRENT_DATE - INTERVAL '90 days'

**Revenue calculation**: Only count orders with status = 'completed'. Use total_amount from orders table, which already accounts for discounts.

**Customer lifetime value (CLV)**: Sum of all completed order amounts for a customer.

**High-value orders**: Orders with total_amount > 1000

## Example Query

-- Get top 10 customers by revenue in the last quarter
SELECT
    c.customer_id,
    c.name,
    c.customer_tier,
    SUM(o.total_amount) as total_revenue
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status = 'completed'
  AND o.order_date >= CURRENT_DATE - INTERVAL '3 months'
GROUP BY c.customer_id, c.name, c.customer_tier
ORDER BY total_revenue DESC
LIMIT 10

必要なライブラリをインポートします。`skills_ref` には agent skill の一覧取得、バリデーション、パースのヘルパー関数が含まれています。これは Anthropic がリリースした[公式リポジトリ](https://github.com/agentskills/agentskills/tree/main)をベースにしています。

In [ ]:
import subprocess
from typing import NotRequired
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, ModelResponse, AgentMiddleware, AgentState
from langchain.messages import SystemMessage
from typing import Callable
from pathlib import Path
from skills_ref.utils import list_skills
from skills_ref.models import SkillProperties

agent が skill のメタデータを保存するために使用する state を定義します

In [ ]:
class SkillsState(AgentState):
    """State for the skills middleware."""

    skills_metadata: NotRequired[list[SkillProperties]]
    """List of loaded skill metadata (name, description)."""

agent がディレクトリ一覧の取得、ファイルの読み込み、SQL クエリの実行を行えるようにする bash コマンド skill を作成します。

In [ ]:
@tool
def bash(command: str) -> str:
    """Execute a bash command and return the output.

    Use this to access skills and resources on the filesystem:
    - Read skill files: cat /path/to/skill/SKILL.md
    - List directories: ls /path/to/dir

    Args:
        command: The bash command to execute
    """
    if not command.startswith(('ls','cat','sqlite3')):
        return 'only "ls","cat" and "sqlite3" commands are allowed'
    result = subprocess.run(command, shell=True, capture_output=True, text=True, timeout=30)
    output = result.stdout
    if result.returncode != 0 and result.stderr:
        output = f"Error (exit {result.returncode}): {result.stderr}\n{output}"
    return output or result.stderr

<table style="width: 100%; table-layout: fixed;">
  <tr>
    <td style="width: 50%; text-align: center; vertical-align: top; padding: 10px;">
      <div>The core agent loop involves calling a model, letting it choose tools to execute, and then finishing when it calls no more tools.</div>
    </td>
    <td style="width: 50%; text-align: center; vertical-align: top; padding: 10px;">
      <div>Middleware exposes hooks before and after each of those steps.</div>
    </td>
  </tr>
  <tr>
     <td style="width: 50%; text-align: center; vertical-align: top; padding: 10px;">
      <img src="./images/core_agent_loop.jpeg" alt="core agent loop" style="max-width: 100%; height: auto; display: block; margin: 10px auto;">
    </td>
    <td style="width: 50%; text-align: center; vertical-align: top; padding: 10px;">
      <img src="./images/middleware_final.jpeg" alt="middleware agent loop" style="max-width: 50%; height: auto; display: block; margin: 10px auto;">
    </td>
  </tr>
</table>

skill の説明をシステムプロンプトに注入するカスタムミドルウェアを作成します。  
Agent Middleware インターフェースで実装できるすべてのプロパティと関数の一覧は[こちら](https://github.com/langchain-ai/langchain/blob/c930062f69bbf72d0147db2e2db1940777966ffe/libs/langchain_v1/langchain/agents/middleware/types.py#L343-L756)を参照してください。  
このチュートリアルとチャレンジでは `state_schema`、`tools`、`before_agent`、`wrap_model_call` のみを使用します。  
`state_schema` には skill のメタデータのリストが含まれています。これにより、skill プロンプトの注入を通じてモデル（LLM）に skill を公開します。  
`tools = [bash]` を指定することで、agent が `bash` tool を使ってディレクトリ一覧の取得、skill の命令の読み込み、SQL クエリの実行を行えるようにします。  
`before_agent` では、システムディレクトリから skill のメタデータ（name、description）を動的に取得し、agent の `state_schema` を更新します。詳細は [utils.py](./skills_ref/utils.py) を参照してください。  
`wrap_model_call` では、agent の状態から skill のメタデータを取得し、skill の付加情報を構築してシステムプロンプトに追加します。

本番環境で動作する agent には追加のガードレールが必要です。  
例：
1) tool の実行は制約されたフォルダー/環境内のスクリプトに限定する。
2) 書き込みコマンドを発行する tool の実行前に承認ステップを設ける。
3) 許可されたコマンドの一覧（できれば読み取り専用のコマンドのみ）を管理する。
4) 機密データへのアクセス制御を実装する（例：Model Context Protocol での OAuth）。
5) 重要なアクションについては human-in-the-loop による承認を行う。

In [ ]:
# Create skill middleware
class SkillMiddleware(AgentMiddleware):
    """Middleware that injects skill descriptions into the system prompt."""

    state_schema = SkillsState

    # Register the bash tool as a class variable
    tools = [bash]

    def __init__(self,skills_dir):
        self.skills_dir = skills_dir

    def before_agent(self, state:SkillsState, runtime):
        skills = list_skills(self.skills_dir)
        return SkillsState(skills_metadata=skills)

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        """Sync: Inject skill descriptions into system prompt."""

        skills = request.state.get("skills_metadata", [])
        skills_list = []
        for skill in skills:
            skills_list.append(
                f"- **{self.skills_dir / skill.name}**: {skill.description}"
            )
        self.skills_prompt = "\n".join(skills_list)

        # Build the skills addendum
        skills_addendum = (
            f"\n\n## Available Skills\n\n{self.skills_prompt}\n\n"
            "Use the bash tool to read skill instructions, e.g., "
            "`bash('cat /path/to/skill/SKILL.md')`. "
        )

        # Append to system message content blocks
        new_content = list(request.system_message.content_blocks) + [
            {"type": "text", "text": skills_addendum}
        ]
        new_system_message = SystemMessage(content=new_content)
        modified_request = request.override(system_message=new_system_message)
        response = handler(modified_request)
        return response

agent を作成する関数を定義します。ここでミドルウェアとレスポンス形式を指定します。

In [ ]:
from pydantic import BaseModel, Field

class SQLOutput(BaseModel):
    sql: str = Field(description="runnable SQL query")

from langchain.chat_models import init_chat_model

def create_sql_agent(skills_dir,inf_url,nvidia_api_key,debug=False):
    model_id = "openai/gpt-oss-120b"
    nvidia_model = init_chat_model(model=model_id,base_url=inf_url,api_key=nvidia_api_key,model_provider="nvidia")
    # Create the agent with skill support
    agent = create_agent(
        nvidia_model,
        system_prompt=(
            f"""
            You are a SQL query assistant that generates runnable SQL query for a sales-analytics database.
            Only return the SQL query and nothing else.
            """
        ),
        middleware=[SkillMiddleware(skills_dir)],
        response_format=SQLOutput,
        debug=debug
    )
    return agent

skills ディレクトリを指定して agent を初期化します。

In [ ]:
skills_dir = Path.cwd().resolve() / 'skills'
inf_url = "https://integrate.api.nvidia.com/v1"
nvidia_api_key = os.environ["NVIDIA_API_KEY"]
agent = create_sql_agent(skills_dir,inf_url,nvidia_api_key,debug=True)

skill の読み込みをトリガーする質問をします。

In [ ]:
result = agent.invoke(  
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Write a SQL query to find all customers "
                    "who made orders over $1000 in the last month"
                ),
            }
        ]
    }
)

# Print the conversation
for message in result["messages"]:
    if hasattr(message, 'pretty_print'):
        message.pretty_print()
    else:
        print(f"{message.type}: {message.content}")

Expected output:

```
================================ Human Message =================================

Write a SQL query to find all customers who made orders over $1000 in the last month
================================== Ai Message ==================================
Tool Calls:
  bash (chatcmpl-tool-ad22f515e713f986)
 Call ID: chatcmpl-tool-ad22f515e713f986
  Args:
    command: cat /Users/chongenl/Documents/agentic-ai-bootcamp/tutorial/jupyter_notebook/skills/sales-analytics
================================= Tool Message =================================
Name: bash

Error (exit 1): cat: /Users/chongenl/Documents/agentic-ai-bootcamp/tutorial/jupyter_notebook/skills/sales-analytics: Is a directory


================================== Ai Message ==================================
Tool Calls:
  bash (chatcmpl-tool-80378d823d869bdf)
 Call ID: chatcmpl-tool-80378d823d869bdf
  Args:
    command: ls -R /Users/chongenl/Documents/agentic-ai-bootcamp/tutorial/jupyter_notebook/skills/sales-analytics
================================= Tool Message =================================
Name: bash

SKILL.md

================================== Ai Message ==================================
Tool Calls:
  bash (chatcmpl-tool-aaa17cbb062a49c7)
 Call ID: chatcmpl-tool-aaa17cbb062a49c7
  Args:
    command: cat /Users/chongenl/Documents/agentic-ai-bootcamp/tutorial/jupyter_notebook/skills/sales-analytics/SKILL.md
================================= Tool Message =================================
Name: bash

---
name: sales-analytics
description: Database schema and business logic for sales data analysis including customers, orders, and revenue.
---
# Sales Analytics Schema

## Tables

### customers
- customer_id (PRIMARY KEY)
- name
- email
- signup_date
- status (active/inactive)
- customer_tier (bronze/silver/gold/platinum)

### orders
- order_id (PRIMARY KEY)
- customer_id (FOREIGN KEY -> customers)
- order_date
- status (pending/completed/cancelled/refunded)
- total_amount
- sales_region (north/south/east/west)

### order_items
- item_id (PRIMARY KEY)
- order_id (FOREIGN KEY -> orders)
- product_id
- quantity
- unit_price
- discount_percent

## Business Logic

**Active customers**: status = 'active' AND signup_date <= CURRENT_DATE - INTERVAL '90 days'

**Revenue calculation**: Only count orders with status = 'completed'. Use total_amount from orders table, which already accounts for discounts.

**Customer lifetime value (CLV)**: Sum of all completed order amounts for a customer.

**High-value orders**: Orders with total_amount > 1000

## Example Query

-- Get top 10 customers by revenue in the last quarter
SELECT
    c.customer_id,
    c.name,
    c.customer_tier,
    SUM(o.total_amount) as total_revenue
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status = 'completed'
  AND o.order_date >= CURRENT_DATE - INTERVAL '3 months'
GROUP BY c.customer_id, c.name, c.customer_tier
ORDER BY total_revenue DESC
LIMIT 10

================================== Ai Message ==================================

{
    "sql": "SELECT DISTINCT c.customer_id, c.name, c.email\nFROM customers c\nJOIN orders o ON c.customer_id = o.customer_id\nWHERE o.status = 'completed'\n  AND o.total_amount > 1000\n  AND o.order_date >= CURRENT_DATE - INTERVAL '1 month';"
}
```

structured output を取得します。

In [ ]:
print(result['structured_response'].sql)

Expected output:

```
SELECT DISTINCT c.customer_id, c.name, c.email
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status = 'completed'
  AND o.total_amount > 1000
  AND o.order_date >= CURRENT_DATE - INTERVAL '1 month';
```

### Agent skills と MCP の比較

| 状況 | skills を優先 | MCP を優先 | 最適な組み合わせ方 |
| ---------------------------------------------------------- | --------------------- | ------------- | -------------------------------------------------------------------------------------------------------------------------------------- |
| より良い動作が必要（ツールは既存のものを使用） | はい | 不要 | 既存のツールを使って agent が計画・確認・結果の整形を行う方法を示す skill を追加する。 |
| 新しいシステムやデータへの接続が必要 | 不十分 | はい | それらのシステム用の MCP ツールを定義し、オプションで各ツールを安全・効果的に使用する方法を説明する skill を追加する。 |
| 製品間で再利用可能なドメイン「プレイブック」が必要 | はい | 任意 | Skill がプレイブックを保持し、ツールが必要な箇所では MCP を使用する。 |
| エンタープライズ統合・セキュリティ・監査性が最優先 | 補助的 | はい | MCP が権限を管理し、skill が内部ポリシーやワークフローを記述する。 |

## 参考リンク

- [LangGraph repo](https://github.com/langchain-ai/langgraph)
- [LangGraph short term memory](https://docs.langchain.com/oss/python/langgraph/add-memory#add-short-term-memory)
- [LangGraph Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts)
- [LangChain Agent Skills](https://docs.langchain.com/oss/python/langchain/multi-agent/skills-sql-assistant)
- [Agent skills open protocol](https://agentskills.io/home)
- [LangChain NVIDIA](https://github.com/langchain-ai/langchain-nvidia)

---

## ライセンス

Copyright © 2026 OpenACC-Standard.org. 本資料は OpenACC-Standard.org が NVIDIA Corporation との協力のもと、Creative Commons Attribution 4.0 International (CC BY 4.0) ライセンスのもとで公開しています。本資料には他の団体が開発したハードウェアおよびソフトウェアへの参照が含まれており、該当するすべてのライセンスおよび著作権が適用されます。

<p> <center> <a href="../start_here_ja.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="03_low_level_mcp_ja.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="01_inference_endpoint_ja.ipynb">1</a>
        <a href="02_introduction_mcp_ja.ipynb">2</a>
        <a href="03_low_level_mcp_ja.ipynb">3</a>
        <a >4</a>
        <a href="05_nemo_agent_toolkit_ja.ipynb">5</a>
        <a href="06_challenge_ja.ipynb">6</a>
        <a href="bonus_challenge/07_bonus_challenge_ja.ipynb">7</a>
    </span>
    <span style="float: left; width: 33%; text-align: right;"><a href="05_nemo_agent_toolkit_ja.ipynb">Next Notebook</a></span>
</div>